In [ ]:
# Adds the scripts folder to the Python path
import sys
sys.path.append("../")  

In [2]:
pip install ngboost scikit-optimize

In [3]:
pip install ngboost

Note: you may need to restart the kernel to use updated packages.


In [ ]:

import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import MLE

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Paths and filenames
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind.joblib")
OUT_TABLE    = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_Predictions.csv")
os.makedirs(os.path.dirname(OUT_TABLE), exist_ok=True)
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# Load dataset and set up features and target
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").drop(columns=["Date"])
X = df.drop(columns=["Wind_GWh"])
y = df["Wind_GWh"]

# Chronological split (no shuffling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Base NGBoost estimator
ngb = NGBRegressor(Dist=Normal, Score=MLE, verbose=False)

# Bayesian search space
search_space = {
    "n_estimators": Integer(300, 700),
    "learning_rate": Real(0.01, 0.1, prior="log-uniform"),
    "minibatch_frac": Real(0.5, 1.0),
    "col_sample": Real(0.8, 1.0),
    "natural_gradient": Categorical([True, False]),
}

# Bayesian optimization with 3-fold CV, optimizing MAE
bayes_search = BayesSearchCV(
    estimator=ngb,
    search_spaces=search_space,
    n_iter=30,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

print("Starting Bayesian hyperparameter tuning...")
t0 = time.time()
bayes_search.fit(X_train, y_train)
tuning_duration = round(time.time() - t0, 2)
print("Best parameters:", bayes_search.best_params_)
print("Tuning time (s):", tuning_duration)

# Train final model with best parameters and save it
final_model = bayes_search.best_estimator_
joblib.dump(final_model, MODEL_PATH)

# Evaluate on the test set
y_pred = final_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE {mae:.2f}, RMSE {rmse:.2f}, R² {r2:.3f}")

# Save predictions table
preds_df = pd.DataFrame(
    {"Actual": y_test.values, "Predicted": y_pred},
    index=y_test.index
)
preds_df.to_csv(OUT_TABLE)

print("NGBoost training and evaluation complete.")
print("Model saved to:", MODEL_PATH)
print("Predictions saved to:", OUT_TABLE)


In [ ]:
from log_utils import log_model_performance, save_log_to_csv


log_dict = log_model_performance(
    model_name="NGBoost_Wind",
    target_variable="Wind_GWh",
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred,
    model_object=final_model,
    training_time_sec=tuning_duration,
    dataset_name="Final_Wind_Data_Model",
    tuning_type="BayesSearchCV",
    gpu_used="NVIDIA GeForce RTX 2050",
    extra_notes="Bayesian tuning with 30 iterations"
)
save_log_to_csv(log_dict)

CRPS

In [ ]:
# CRPS evaluation for NGBoost (Wind Model)

import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import norm
from sklearn.model_selection import train_test_split

# Paths
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind.joblib")
OUT_CSV      = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_CRPS.csv")
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# Load dataset
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date")

# Keep Date index for reporting
dates = df["Date"]

# Features and target
X = df.drop(columns=["Date", "Wind_GWh"])
y = df["Wind_GWh"].astype(float)

# Chronological split (same as training)
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, shuffle=False
)

# Load trained NGBoost model
ngb = joblib.load(MODEL_PATH)

# Predictive distribution for the test set
dist = ngb.pred_dist(X_test)

# Extract mean and standard deviation
mu = getattr(dist, "loc", None)
sigma = getattr(dist, "scale", None)

if mu is None or sigma is None:
    params = getattr(dist, "params", None)
    if params is not None:
        if isinstance(params, dict):
            mu = params.get("loc", params.get("mean", None))
            sigma = params.get("scale", params.get("std", None))
        else:
            try:
                mu, sigma = params
            except Exception:
                pass

if mu is None:
    try: mu = dist.mean()
    except Exception: pass

if sigma is None:
    try: sigma = dist.std()
    except Exception: pass

mu = np.asarray(mu, dtype=float)
sigma = np.asarray(sigma, dtype=float)
sigma = np.maximum(sigma, 1e-8)  # prevent zero or negative std

# Compute CRPS for Normal
# CRPS(N(μ,σ), y) = σ * [1/√π - 2φ(z) - z(2Φ(z)-1)], where z = (y-μ)/σ
y_true = y_test.values.astype(float)
z = (y_true - mu) / sigma
phi = norm.pdf(z)
Phi = norm.cdf(z)
crps = sigma * ((1.0 / np.sqrt(np.pi)) - 2.0 * phi - z * (2.0 * Phi - 1.0))

# Save results table
crps_df = pd.DataFrame({
    "Date": dates_test.values,
    "Actual": y_true,
    "Predicted_Mean": mu,
    "Predicted_Std": sigma,
    "CRPS": crps
}).set_index("Date")

crps_df.to_csv(OUT_CSV)

print(f"CRPS table saved to {OUT_CSV}")
print(f"Mean CRPS: {crps_df['CRPS'].mean():.4f}")


NGBoost Actual vs Predicted line plot

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import os
import joblib

# Path
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind.joblib")
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model.csv")
TABLE_PATH = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_actual_vs_predicted.csv")
PLOT_PATH = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_shap","NGBoost_actual_vs_predicted.png")

# Load data
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").drop(columns=["Date"])
X = df.drop(columns=["Wind_GWh"])
y = df["Wind_GWh"]

# Chronological 80/20 split
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Load model
model = joblib.load(MODEL_PATH)

# Predict
y_pred = model.predict(X_test)

# save actual vs predicted table
results_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
}, index=y_test.index)

os.makedirs(os.path.dirname(TABLE_PATH), exist_ok=True)
results_df.to_csv(TABLE_PATH, index_label="Index")
print(f" Saved table: {TABLE_PATH}")

# === PLOT ===
plt.figure(figsize=(12, 6))
plt.plot(results_df["Actual"], label="Actual Wind_GWh", color="#000000", linewidth=2)
plt.plot(results_df["Predicted"], label="Predicted Wind_GWh", color="#00b4a2", linestyle="--", linewidth=2)
plt.title("NGBoost – Actual vs Predicted Wind_GWh")
plt.xlabel("Test Sample Index")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Save plot
os.makedirs(os.path.dirname(PLOT_PATH), exist_ok=True)
plt.savefig(PLOT_PATH, dpi=300)
plt.show()
print(f"✅ Saved plot: {PLOT_PATH}")


In [ ]:
# NGBoost Wind: predictions with calibrated 90% confidence intervals

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind.joblib")
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model.csv")
TABLE_PATH   = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_with_CI.csv")
PLOT_PATH    = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_shap", "NGBoost_Wind_CI.png")

# Load data and set up features/target
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date")
X = df.drop(columns=["Wind_GWh", "Date"])
y = df["Wind_GWh"]
dates = df["Date"]

# Chronological 80/20 split
split_idx  = int(len(df) * 0.8)
X_train    = X.iloc[:split_idx]
X_test     = X.iloc[split_idx:]
y_train    = y.iloc[:split_idx]
y_test     = y.iloc[split_idx:]
dates_test = dates.iloc[split_idx:]

# Load trained model
model = joblib.load(MODEL_PATH)

# Point predictions on the test set
y_pred = model.predict(X_test)

# Calibrate a 90% interval using train residual quantiles
train_pred = model.predict(X_train)
residuals  = y_train - train_pred
lower_q    = np.quantile(residuals, 0.05)  # 5th percentile
upper_q    = np.quantile(residuals, 0.95)  # 95th percentile

lower_bound = y_pred + lower_q
upper_bound = y_pred + upper_q

# Coverage of the calibrated interval on the test set
coverage = np.mean((y_test >= lower_bound) & (y_test <= upper_bound))
print(f"90% CI coverage accuracy: {coverage:.3f}")

# Save results table
results_df = pd.DataFrame({
    "Date": dates_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Lower_90": lower_bound,
    "Upper_90": upper_bound
})
os.makedirs(os.path.dirname(TABLE_PATH), exist_ok=True)
results_df.to_csv(TABLE_PATH, index=False)
print("Saved CI results table:", TABLE_PATH)

# Plot actual vs prediction with calibrated intervals
plt.figure(figsize=(12, 6))
plt.plot(dates_test, y_test, label="Actual", color="#015d5c", linewidth=2)
plt.plot(dates_test, y_pred, label="Predicted", color="#00b4a2", linestyle="--", linewidth=2)
plt.fill_between(dates_test, lower_bound, upper_bound, color="#00b4a2", alpha=0.2, label="90% CI")
plt.title("NGBoost — Predictions with 90% Confidence Interval")
plt.xlabel("Date")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

os.makedirs(os.path.dirname(PLOT_PATH), exist_ok=True)
plt.savefig(PLOT_PATH, dpi=300)
plt.show()
print("Saved CI plot:", PLOT_PATH)


In [ ]:
# ngboost_wind_ci_calibrated_index.py
# Calibrate NGBoost predictive intervals using standardized residuals (index-based plot)

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind.joblib")
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model.csv")
PLOT_PATH    = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_Wind_CI_index_CALIBRATED.png")
TABLE_PATH   = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_with_CI_CALIBRATED.csv")

# Load data and split chronologically (80/20)
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
X = df.drop(columns=["Wind_GWh", "Date"])
y = df["Wind_GWh"]

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Load trained model
model = joblib.load(MODEL_PATH)

# Predictive distributions on train and test
dist_tr = model.pred_dist(X_train)
mu_tr, sig_tr = dist_tr.loc, dist_tr.scale

dist_te = model.pred_dist(X_test)
mu_te, sig_te = dist_te.loc, dist_te.scale

# Calibrate using standardized residuals on the training set
std_res = (y_train.values - mu_tr) / (sig_tr + 1e-12)  # avoid division by zero
t90 = np.quantile(np.abs(std_res), 0.90)               # two-sided 90% threshold
t95 = np.quantile(np.abs(std_res), 0.95)               # optional 95% threshold

# Calibrated 90% intervals on the test set
pred  = mu_te
low90 = mu_te - t90 * sig_te
upp90 = mu_te + t90 * sig_te
# Optional 95%:
# low95 = mu_te - t95 * sig_te
# upp95 = mu_te + t95 * sig_te

# Coverage on the test set
cov90 = float(np.mean((y_test.values >= low90) & (y_test.values <= upp90)))
print(f"Calibrated 90% CI coverage: {cov90:.3f}")

# Save table (index-based)
os.makedirs(os.path.dirname(TABLE_PATH), exist_ok=True)
pd.DataFrame(
    {
        "Actual": y_test.values,
        "Predicted": pred,
        "Lower90": low90,
        "Upper90": upp90,
        # "Lower95": low95,
        # "Upper95": upp95,
    },
    index=y_test.index
).to_csv(TABLE_PATH, index_label="Index")
print("Saved calibrated CI table:", TABLE_PATH)

# Plot with index on the x-axis
idx = np.arange(len(y_test))
plt.figure(figsize=(12, 6))
plt.plot(idx, y_test.values, label="Actual", color="#000000", linewidth=2)
plt.plot(idx, pred,         label="Predicted", color="#1f77b4", linestyle="--", linewidth=2)
plt.fill_between(idx, low90, upp90, color="#aec7e8", alpha=0.3, label="90% CI (calibrated)")
plt.title("NGBoost: Predictions with 90% Confidence Interval (Index Axis, Calibrated)")
plt.xlabel("Test Sample Index")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

os.makedirs(os.path.dirname(PLOT_PATH), exist_ok=True)
plt.savefig(PLOT_PATH, dpi=300, bbox_inches="tight")
plt.show()
print("Saved calibrated CI plot:", PLOT_PATH)

SHAP Summary Bar + Beeswarm Plot for NGBoost

In [ ]:
# SHAP analysis for NGBoost mean prediction using a high-fidelity tree surrogate

import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score

# Configuration
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"

# Use the NGBoost model saved for results
MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind.joblib")
DATA_PATH  = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model.csv")

SAVE_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_shap")
os.makedirs(SAVE_DIR, exist_ok=True)

BAR_PATH       = os.path.join(SAVE_DIR, "shap_bar.png")
BEESWARM_PATH  = os.path.join(SAVE_DIR, "shap_beeswarm.png")
WATERFALL_PATH = os.path.join(SAVE_DIR, "shap_waterfall.png")
TOP10_CSV      = os.path.join(SAVE_DIR, "top_10_features.csv")

TEAL = "#00b4a2"
DARK = "#015d5c"

# Load data and split chronologically (80/20)
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
X = df.drop(columns=["Date", "Wind_GWh"])
y = df["Wind_GWh"]

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Load NGBoost and get mean predictions (μ)
ngb = joblib.load(MODEL_PATH)

mu_train = ngb.pred_dist(X_train).loc
mu_test  = ngb.pred_dist(X_test).loc

# Train surrogate tree to mimic NGBoost mean
surrogate = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
surrogate.fit(X_train, mu_train)

# Measure surrogate fidelity on the test set
mu_test_hat = surrogate.predict(X_test)
fidelity_r2 = r2_score(mu_test, mu_test_hat)
print(f"Surrogate fidelity to NGBoost mean (test R^2): {fidelity_r2:.3f}")

# SHAP on the surrogate (TreeExplainer)
explainer = shap.TreeExplainer(surrogate)
shap_values = explainer(X_test)

# Bar plot (global mean absolute SHAP)
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=15, show=False)
ax = plt.gca()
for p in getattr(ax, "patches", []):
    p.set_facecolor(TEAL)
plt.title("SHAP Summary (NGBoost mean via surrogate)")
plt.tight_layout()
plt.savefig(BAR_PATH, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", BAR_PATH)

# Waterfall plot for a representative test sample
sample_idx = min(5, len(X_test) - 1)  # safe index within range
plt.figure(figsize=(9, 7))
shap.plots.waterfall(shap_values[sample_idx], max_display=12, show=False)
plt.title(f"SHAP Waterfall – Test sample #{sample_idx} (NGBoost mean via surrogate)")
plt.tight_layout()
plt.savefig(WATERFALL_PATH, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", WATERFALL_PATH)

# Top-10 features table based on mean absolute SHAP
if hasattr(shap_values, "values"):
    sv = np.abs(shap_values.values)
else:
    sv = np.abs(shap_values)  # fallback for older SHAP versions

mean_abs = sv.mean(axis=0)
std_abs  = sv.std(axis=0)

top_df = (
    pd.DataFrame({"Feature": X_test.columns, "Mean_SHAP": mean_abs, "Std_Dev": std_abs})
      .sort_values("Mean_SHAP", ascending=False)
      .head(10)
      .reset_index(drop=True)
)
top_df.index = np.arange(1, len(top_df) + 1)
top_df.to_csv(TOP10_CSV, index_label="Rank")
print("Saved top-10 features:", TOP10_CSV)


In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd

from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import MLE

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Configuration
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind_Pruned_features.joblib")
PRED_PATH    = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_Pruned_Predictions.csv")

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
os.makedirs(os.path.dirname(PRED_PATH), exist_ok=True)

TEAL = "#00b4a2"  # unused color placeholder

# Load data
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

# Drop low-importance features if present
drop_cols = ["Wind_GWh_Lag3", "Wind_GWh_Lag6", "Cloud_Cover"]
drop_cols = [c for c in drop_cols if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

# Features and target
TARGET = "Wind_GWh"
X = df.drop(columns=["Date", TARGET])
y = df[TARGET]

# Chronological split 80/20
split_idx = int(len(df) * 0.80)
X_trainval, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_trainval, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df["Date"].iloc[split_idx:]

# Base estimator
ngb = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    verbose=False,
    random_state=42
)

# Search space (bounds chosen for expanded lags)
search_space = {
    "n_estimators": Integer(500, 900),
    "learning_rate": Real(0.01, 0.04, prior="log-uniform"),
    "minibatch_frac": Real(0.6, 0.9),
    "col_sample": Real(0.85, 1.0),
    "natural_gradient": Categorical([True])
}

# Leakage-safe cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Bayesian tuning
print("Tuning NGBoost (expanded lags, pruned features)...")
t0 = time.time()
search = BayesSearchCV(
    estimator=ngb,
    search_spaces=search_space,
    n_iter=40,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42,
    verbose=0
)
search.fit(X_trainval, y_trainval)
tuning_time = round(time.time() - t0, 2)
best_params = search.best_params_
print("Best parameters:", best_params)

# Final refit with early stopping on a small validation tail
val_tail = max(1, int(len(X_trainval) * 0.10))
X_train, X_val = X_trainval.iloc[:-val_tail], X_trainval.iloc[-val_tail:]
y_train, y_val = y_trainval.iloc[:-val_tail], y_trainval.iloc[-val_tail:]

final_model = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    verbose=False,
    random_state=42,
    **best_params
)

t1 = time.time()
final_model.fit(X_train, y_train, X_val=X_val, Y_val=y_val, early_stopping_rounds=50)
train_time = round(time.time() - t1, 2)

joblib.dump(final_model, MODEL_PATH)
print("Model saved:", MODEL_PATH)

# Evaluation on the held-out test set
y_pred = final_model.pred_dist(X_test).loc
mae  = mean_absolute_error(y_test, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
r2   = r2_score(y_test, y_pred)

# Save predictions aligned with Date
pd.DataFrame({
    "Date": dates_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred
}).to_csv(PRED_PATH, index=False)

print("\nNGBoost (expanded lags, pruned) - test metrics")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R^2:  {r2:.3f}")
print(f"Tuning time (s): {tuning_time} | Final refit time (s): {train_time}")
print("Predictions saved to:", PRED_PATH)


In [ ]:

import os
import joblib
import pandas as pd
from log_utils import log_model_performance, save_log_to_csv

# Config
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind_Pruned_features.joblib")

# Load dataset
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

# Drop low-importance features (same as training script)
drop_cols = ["Wind_GWh_Lag3", "Wind_GWh_Lag6", "Cloud_Cover"]
drop_cols = [c for c in drop_cols if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

# Features & Target
TARGET = "Wind_GWh"
X = df.drop(columns=["Date", TARGET])
y = df[TARGET]

# Chronological 80/20 split
split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Load model & predict
final_model = joblib.load(MODEL_PATH)
y_pred = final_model.predict(X_test)

# log and csv
log_dict = log_model_performance(
    model_name="NGBoost_Wind_Pruned_features",
    target_variable="Wind_GWh",
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred,
    model_object=final_model,
    training_time_sec=0.52,   
    dataset_name="Final_Wind_Data_Model_With_Expanded_Lags",
    tuning_type="BayesSearchCV (40 iterations)",
    gpu_used="NVIDIA GeForce RTX 2050",
    extra_notes="Expanded lags with pruned low-SHAP features; tuned over 40 iterations"
)

save_log_to_csv(log_dict)
print(" Log entry saved for NGBoost_Wind_Pruned_1")


ngboost_wind_actual_vs_predicted

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# config
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind_Pruned_features.joblib")
TABLE_PATH   = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_actual_vs_predicted.csv")
PLOT_PATH    = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_Wind_Pruned", "NGBoost_actual_vs_predicted.png")

# Create directories
os.makedirs(os.path.dirname(TABLE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(PLOT_PATH), exist_ok=True)

# load data
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
drop_cols = ["Wind_GWh_Lag3", "Wind_GWh_Lag6", "Cloud_Cover"]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

X = df.drop(columns=["Date", "Wind_GWh"])
y = df["Wind_GWh"]

# Split 80/20
split_idx = int(len(df) * 0.8)
X_test = X.iloc[split_idx:]
y_test = y.iloc[split_idx:]

# load model and predict
model = joblib.load(MODEL_PATH)
y_pred = model.predict(X_test)

# save table
results_df = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred}, index=y_test.index)
results_df.to_csv(TABLE_PATH, index_label="Index")
print(f" Saved table: {TABLE_PATH}")

# plot
plt.figure(figsize=(12, 6))
plt.plot(results_df["Actual"], label="Actual", color="#015d5c", linewidth=2)
plt.plot(results_df["Predicted"], label="Predicted", color="#00b4a2", linestyle="--", linewidth=2)
plt.title("NGBoost_Wind_Pruned — Actual vs Predicted")
plt.xlabel("Test Sample Index")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=300)
plt.show()
print(f" Saved plot: {PLOT_PATH}")


ngboost_wind_coverage

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import numpy as np

# config
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind_Pruned_features.joblib")
TABLE_PATH   = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Wind_pruned_CI_Coverage.csv")
PLOT_PATH    = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_Wind_Pruned", "NGBoost_pruned_CI_coverage.png")

os.makedirs(os.path.dirname(TABLE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(PLOT_PATH), exist_ok=True)

# load data
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
drop_cols = ["Wind_GWh_Lag3", "Wind_GWh_Lag6", "Cloud_Cover"]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)

X = df.drop(columns=["Date", "Wind_GWh"])
y = df["Wind_GWh"]

split_idx = int(len(df) * 0.8)
X_test = X.iloc[split_idx:]
y_test = y.iloc[split_idx:]
dates_test = df["Date"].iloc[split_idx:]

# load model
model = joblib.load(MODEL_PATH)

# Predictive distribution
pred_dist = model.pred_dist(X_test)
mean_pred = pred_dist.loc
lower_90 = pred_dist.ppf(0.05)
upper_90 = pred_dist.ppf(0.95)

# Coverage
coverage = np.mean((y_test >= lower_90) & (y_test <= upper_90))
print(f" 90% CI Coverage: {coverage:.3f}")

# Save table
ci_df = pd.DataFrame({
    "Date": dates_test.values,
    "Actual": y_test.values,
    "Predicted": mean_pred,
    "Lower90": lower_90,
    "Upper90": upper_90
})
ci_df.to_csv(TABLE_PATH, index=False)
print(f" Saved table: {TABLE_PATH}")

# Plot
plt.figure(figsize=(12, 6))
plt.plot(dates_test, y_test, label="Actual", color="#015d5c")
plt.plot(dates_test, mean_pred, label="Predicted", color="#00b4a2")
plt.fill_between(dates_test, lower_90, upper_90, color="#00b4a2", alpha=0.2, label="90% CI")
plt.title(f"NGBoost_Wind_Pruned — 90% CI Coverage ({coverage:.1%})")
plt.xlabel("Date")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=300)
plt.show()
print(f" Saved plot: {PLOT_PATH}")


ngboost_wind_shap

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor

# config
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Wind_Pruned_features.joblib")
SAVE_DIR     = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_Wind_Pruned")

TEAL = "#00b4a2"
os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

drop_cols = ["Wind_GWh_Lag3", "Wind_GWh_Lag6", "Cloud_Cover"]
drop_cols = [c for c in drop_cols if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

TARGET = "Wind_GWh"
X = df.drop(columns=["Date", TARGET])
y = df[TARGET]

# Chronological split
split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Load Ngboost model
ngb = joblib.load(MODEL_PATH)

# surrogate: fit ExtraTrees to mimic NGBoost mean predictions on train
y_ngb_train = ngb.predict(X_train)  # NGBoost mean
surrogate = ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)
surrogate.fit(X_train, y_ngb_train)

# SHAP on the surrogate tree model 
explainer = shap.TreeExplainer(surrogate)
shap_values = explainer.shap_values(X_test) 

# GLOBAL IMPORTANCE (mean |SHAP|) 
mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": mean_abs})
shap_df = shap_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

# Save Top 10 CSV
top10_path = os.path.join(SAVE_DIR, "ngboost_wind_pruned_top_10_features.csv")
shap_df.head(10).to_csv(top10_path, index=False)
print(f" Saved Top 10 features: {top10_path}")

# BAR PLOT (Top 20, teal)
top_k = 20
plot_df = shap_df.head(top_k).iloc[::-1] 
plt.figure(figsize=(10, 7))
plt.barh(plot_df["feature"], plot_df["mean_abs_shap"], color=TEAL)
plt.title("SHAP — Top Features (Surrogate of NGBoost_Wind_Pruned)")
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
bar_path = os.path.join(SAVE_DIR, "shap_bar.png")
plt.savefig(bar_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f" Saved bar plot: {bar_path}")


# WATERFALL for one test sample (index 0) 
i = 0
exp = shap.Explanation(
    values=shap_values[i],
    base_values=explainer.expected_value,
    data=X_test.iloc[i],
    feature_names=X_test.columns.tolist()
)
plt.figure(figsize=(9, 7.5))
shap.plots.waterfall(exp, max_display=12, show=False)
plt.tight_layout()
waterfall_path = os.path.join(SAVE_DIR, "shap_waterfall.png")
plt.savefig(waterfall_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f" Saved waterfall plot: {waterfall_path}")


In [ ]:

# Final NGBoost attempt: minimal features + seasonality + strong regularization

import os, time, joblib, numpy as np, pandas as pd
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor
from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import MLE
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Configuration
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")

MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_wind_core_features.joblib")
PRED_PATH    = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_wind_core_features.csv")

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
os.makedirs(os.path.dirname(PRED_PATH), exist_ok=True)

TARGET = "Wind_GWh"

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# Load data
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

# Seasonality (month sine/cosine)
df["month"] = df["Date"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

# Feature selection (robust to column name variants)
def pick_existing(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

feat_lag1   = pick_existing(["Wind_GWh_Lag1", "Wind_GWh_lag1", "Wind_GWh_l1"])
feat_lag12  = pick_existing(["Wind_GWh_Lag12", "Wind_GWh_lag12", "Wind_GWh_l12"])
feat_roll3  = pick_existing(["RollingMean_3", "Wind_GWh_RollingMean3", "Wind_GWh_Roll3"])
feat_wspeed = pick_existing(["Wind_Speed_10m", "Wind_Speed", "Wind_Speed_mps"])
feat_cap    = pick_existing(["Wind_Capacity_MW", "Capacity_MW"])

core_feats = [f for f in [feat_lag1, feat_lag12, feat_roll3, feat_wspeed, feat_cap] if f is not None]
seasonal_feats = ["month_sin", "month_cos"]

if len(core_feats) < 3:
    warnings.warn(f"Only found {len(core_feats)} core features: {core_feats}. Consider checking column names.")

X = df[core_feats + seasonal_feats].copy()
y = df[TARGET].copy()

# Chronological split (80/20)
split_idx = int(len(df) * 0.80)
X_trainval, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_trainval, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df["Date"].iloc[split_idx:]

# NGBoost with strong regularization
BaseTree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=6, random_state=42)
ngb = NGBRegressor(Dist=Normal, Score=MLE, Base=BaseTree, verbose=False, random_state=42)

# Tighter search bounds (small model)
search_space = {
    "n_estimators":     Integer(300, 500),
    "learning_rate":    Real(0.010, 0.030, prior="log-uniform"),
    "minibatch_frac":   Real(0.70, 1.00),
    "col_sample":       Real(0.70, 0.95),
    "natural_gradient": Categorical([True]),
}
tscv = TimeSeriesSplit(n_splits=3)

print("Tuning NGBoost (simplified features, strong regularization)...")
t0 = time.time()
bayes = BayesSearchCV(
    estimator=ngb,
    search_spaces=search_space,
    n_iter=25,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42,
    verbose=0
)
bayes.fit(X_trainval, y_trainval)
tuning_time = round(time.time() - t0, 2)
print("Best Params:", bayes.best_params_)

# Final refit with early stopping on validation tail
val_tail = max(1, int(len(X_trainval) * 0.12))
X_train, X_val = X_trainval.iloc[:-val_tail], X_trainval.iloc[-val_tail:]
y_train, y_val = y_trainval.iloc[:-val_tail], y_trainval.iloc[-val_tail:]

final_model = NGBRegressor(
    Dist=Normal,
    Score=MLE,
    Base=BaseTree,
    verbose=False,
    random_state=42,
    **bayes.best_params_
)

t1 = time.time()
final_model.fit(X_train, y_train, X_val=X_val, Y_val=y_val, early_stopping_rounds=50)
refit_time = round(time.time() - t1, 2)
joblib.dump(final_model, MODEL_PATH)
print("Model saved:", MODEL_PATH)

# Evaluation
y_pred = final_model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
r    = rmse(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

pd.DataFrame(
    {"Date": dates_test.values, "Actual": y_test.values, "Predicted": y_pred}
).to_csv(PRED_PATH, index=False)

print("\nNGBoost (Simplified) — Test Metrics")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {r:.2f}")
print(f"R^2:  {r2:.3f}")
print(f"Tuning time (s): {tuning_time} | Refit+ES time (s): {refit_time}")
print("Predictions saved to:", PRED_PATH)


In [ ]:
from log_utils import log_model_performance, save_log_to_csv

# Aggregate wall time from Block 1
training_time_total = float(tuning_time) + float(refit_time)

model_name = "NGBoost_wind_core_features"

log_dict = log_model_performance(
    model_name=model_name,
    target_variable=TARGET,
    X_train=X_trainval,                # uses the same split objects from Block 1
    X_test=X_test,
    y_test=y_test.values,
    y_pred=y_pred,
    model_object=final_model,
    training_time_sec=training_time_total,
    dataset_name=os.path.basename(DATA_PATH),
    tuning_type=f"BayesSearchCV (25 iters, TimeSeriesSplit=3), early stopping tail={val_tail}",
    gpu_used="NVIDIA GeForce RTX 2050",
    extra_notes=f"Features used: {core_feats + seasonal_feats}; split_idx={int(len(df)*0.80)}; seed=42"
)

save_log_to_csv(log_dict)
print("Logged run to CSV for:", model_name)


ngboost_wind_actual_vs_predicted.

In [ ]:
# Block 3 — Figures and table (reuse objects from Block 1)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_DIR   = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_wind_core_features")
TABLE_DIR = os.path.join(PROJECT_ROOT, "outputs", "results_tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

TABLE_PATH = os.path.join(TABLE_DIR, "NGBoost_Wind_actual_vs_predicted_core_features.csv")
PLOT_PATH  = os.path.join(FIG_DIR, "NGBoost_actual_vs_predicted.png")

# Build and save results table
out_df = pd.DataFrame({
    "Date": dates_test.values,          # from Block 1
    "Actual": y_test.values,            # from Block 1
    "Predicted": y_pred                 # from Block 1
})
out_df.to_csv(TABLE_PATH, index=False)
print("Saved table:", TABLE_PATH)

# Plot actual vs predicted on a date x-axis
plt.figure(figsize=(12, 6))
plt.plot(dates_test.values, y_test.values, label="Actual", color="#015d5c", linewidth=2)
plt.plot(dates_test.values, y_pred,       label="Predicted", color="#00b4a2", linestyle="--", linewidth=2)
plt.title("NGBoost_wind_core_features — Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=300)
plt.show()
print("Saved plot:", PLOT_PATH)


ngboost_wind_coverage

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuration
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_wind_core_features.joblib")

TABLE_PATH   = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_wind_core_features_CI_coverage.csv")
PLOT_PATH    = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_wind_core_features", "NGBoost_CI_coverage.png")

os.makedirs(os.path.dirname(TABLE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(PLOT_PATH), exist_ok=True)

TARGET = "Wind_GWh"

# Load and preprocess data (match training)
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df["month"] = df["Date"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

def pick_existing(cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

feat_lag1   = pick_existing(["Wind_GWh_Lag1", "Wind_GWh_lag1", "Wind_GWh_l1"])
feat_lag12  = pick_existing(["Wind_GWh_Lag12", "Wind_GWh_lag12", "Wind_GWh_l12"])
feat_roll3  = pick_existing(["RollingMean_3", "Wind_GWh_RollingMean3", "Wind_GWh_Roll3"])
feat_wspeed = pick_existing(["Wind_Speed_10m", "Wind_Speed", "Wind_Speed_mps"])
feat_cap    = pick_existing(["Wind_Capacity_MW", "Capacity_MW"])

core_feats = [f for f in [feat_lag1, feat_lag12, feat_roll3, feat_wspeed, feat_cap] if f is not None]
seasonal_feats = ["month_sin", "month_cos"]

X = df[core_feats + seasonal_feats].copy()
y = df[TARGET].copy()
dates = df["Date"]

# Chronological 80/20 split
split_idx = int(len(df) * 0.80)
X_trainval, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_trainval, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test         = dates.iloc[split_idx:]

# Load model
model = joblib.load(MODEL_PATH)

# Predictive distribution on the test set
pred_dist_test = model.pred_dist(X_test)
mean_test  = pred_dist_test.loc
low90_raw  = pred_dist_test.ppf(0.05)  # computed but not plotted
up90_raw   = pred_dist_test.ppf(0.95)  # computed but not plotted

# Recalibrate intervals using residuals on a calibration slice (last 10% of trainval)
cal_tail = max(1, int(len(X_trainval) * 0.10))
X_cal, y_cal = X_trainval.iloc[-cal_tail:], y_trainval.iloc[-cal_tail:]
mean_cal = model.pred_dist(X_cal).loc
residuals = y_cal.values - mean_cal

q_low, q_high = np.quantile(residuals, [0.05, 0.95])

# Calibrated 90% intervals on the test set
lower_90 = mean_test + q_low
upper_90 = mean_test + q_high

# Coverage on the test set
coverage = np.mean((y_test.values >= lower_90) & (y_test.values <= upper_90))
print(f"Calibrated 90% CI coverage: {coverage:.3f}")

# Save table
ci_df = pd.DataFrame({
    "Date": dates_test.values,
    "Actual": y_test.values,
    "Predicted": mean_test,
    "Lower90": lower_90,
    "Upper90": upper_90
})
ci_df.to_csv(TABLE_PATH, index=False)
print("Saved table:", TABLE_PATH)

# Plot calibrated intervals against actuals
plt.figure(figsize=(12, 6))
plt.plot(dates_test, y_test.values, label="Actual", color="#015d5c")
plt.plot(dates_test, mean_test,      label="Predicted", color="#00b4a2")
plt.fill_between(dates_test, lower_90, upper_90, color="#00b4a2", alpha=0.2, label="90% CI (calibrated)")
plt.title(f"NGBoost_Wind_Simplified: 90% CI Coverage ({coverage:.1%})")
plt.xlabel("Date")
plt.ylabel("Wind_GWh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=300)
plt.show()
print("Saved plot:", PLOT_PATH)


ngboost_wind_shap.py (Surrogate-based for NGBoost)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor

# === CONFIG ===
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
DATA_PATH    = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Wind_Data_Model_With_Expanded_Lags.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_wind_core_features.joblib")
SAVE_DIR     = os.path.join(PROJECT_ROOT, "outputs", "figures", "NGBoost_wind_core_features")
os.makedirs(SAVE_DIR, exist_ok=True)

TEAL = "#00b4a2"
TARGET = "Wind_GWh"

# load and preprocess
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
df["month"] = df["Date"].dt.month
df["month_sin"] = np.sin(2*np.pi*df["month"]/12.0)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12.0)

def pick_existing(cands):
    for c in cands:
        if c in df.columns: return c
    return None

feat_lag1   = pick_existing(["Wind_GWh_Lag1", "Wind_GWh_lag1", "Wind_GWh_l1"])
feat_lag12  = pick_existing(["Wind_GWh_Lag12", "Wind_GWh_lag12", "Wind_GWh_l12"])
feat_roll3  = pick_existing(["RollingMean_3", "Wind_GWh_RollingMean3", "Wind_GWh_Roll3"])
feat_wspeed = pick_existing(["Wind_Speed_10m", "Wind_Speed", "Wind_Speed_mps"])
feat_cap    = pick_existing(["Wind_Capacity_MW", "Capacity_MW"])

core_feats = [f for f in [feat_lag1, feat_lag12, feat_roll3, feat_wspeed, feat_cap] if f is not None]
seasonal_feats = ["month_sin", "month_cos"]

X = df[core_feats + seasonal_feats].copy()
y = df[TARGET].copy()

# split
split_idx = int(len(df)*0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]

# Load Ngboost & Build SUrrogate
ngb = joblib.load(MODEL_PATH)
y_ngb_train = ngb.predict(X_train)

surrogate = ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)
surrogate.fit(X_train, y_ngb_train)

# SHAP on surrogate
explainer = shap.TreeExplainer(surrogate)
shap_values = explainer.shap_values(X_test)  # (n_samples, n_features)

# Global importance
mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({"feature": X_test.columns, "mean_abs_shap": mean_abs}) \
          .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

top10_path = os.path.join(SAVE_DIR, "top_10_features.csv")
shap_df.head(10).to_csv(top10_path, index=False)
print(f" Saved Top 10 features: {top10_path}")

# Bar
top_k = 20
plot_df = shap_df.head(top_k).iloc[::-1]
plt.figure(figsize=(10,7))
plt.barh(plot_df["feature"], plot_df["mean_abs_shap"], color=TEAL)
plt.title("SHAP — Top Features (Surrogate of NGBoost_Wind_Simplified)")
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
bar_path = os.path.join(SAVE_DIR, "shap_bar.png")
plt.savefig(bar_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f" Saved bar plot: {bar_path}")


# Waterfall for one sample
i = 0
exp = shap.Explanation(
    values=shap_values[i],
    base_values=explainer.expected_value,
    data=X_test.iloc[i],
    feature_names=X_test.columns.tolist()
)
plt.figure(figsize=(9,7.5))
shap.plots.waterfall(exp, max_display=12, show=False)
plt.tight_layout()
waterfall_path = os.path.join(SAVE_DIR, "shap_waterfall.png")
plt.savefig(waterfall_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f" Saved waterfall plot: {waterfall_path}")
